# scANVI on Object A

* **Developed by:** Anna Maguza
* **Affilation:** CellZome, a GSK company
* **Created date:** 2026-05-08
* **Last modified date:** 2026-05-13

Label-aware scANVI integration on cell_states.


## obj_a_scanvi

Object A — model scanvi (label-aware on cell_states).

**Kernel: `lgr5_enhanced`**.

In [ ]:
import scanpy as sc, anndata as ad, numpy as np, torch, scvi
from pathlib import Path
import matplotlib.pyplot as plt
DATA_OUT = Path('/Users/am336941/PhD/data/Fetal_stem_cells_analysis_enhanced/')

In [19]:
print(f"PyTorch version: {torch.__version__}")
print(f"scVI version: {scvi.__version__}")

PyTorch version: 2.10.0
scVI version: 1.4.2


In [ ]:
INPUT = Path('/Users/am336941/PhD/data/Fetal_stem_cells_analysis_enhanced/integrated_obj_a_scvi.h5ad')
adata = sc.read_h5ad(INPUT)
adata

In [35]:
REPO     = Path('/Users/am336941/Library/CloudStorage/OneDrive-GSK/Desktop/Fetal_stem_cells_analysis')
FIG_DIR  = REPO / 'analysis_enhanced/3_integration/3a_human_only/figures/obj_a_scanvi'
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [20]:
original_load = torch.load

def safe_load_override(*args, **kwargs):
    kwargs['weights_only'] = False
    return original_load(*args, **kwargs)

torch.load = safe_load_override

In [24]:
scvi_model = scvi.model.SCVI.load(f"/Users/am336941/PhD/data/Fetal_stem_cells_analysis_enhanced/model_scvi_obj_a_scvi", adata)

INFO     File /Users/am336941/PhD/data/Fetal_stem_cells_analysis_enhanced/model_scvi_obj_a_scvi/model.pt already   
         downloaded                                                                                                


In [25]:
torch.load = original_load

In [26]:
def _accelerator():
    try:
        import torch
        return 'mps' if torch.backends.mps.is_available() else 'cpu'
    except Exception:
        return 'cpu'
ACCEL = _accelerator()
print('using accelerator:', ACCEL)


using accelerator: mps


In [27]:
scanvi_model = scvi.model.SCANVI.from_scvi_model(scvi_model, 'Unknown')

In [29]:
scanvi_model.train(
    100,
    early_stopping=True,
    check_val_every_n_epoch=1,
    enable_progress_bar=True, accelerator="mps",
    #plan_kwargs={
    #    "lr": 1e-6,  # Reduced learning rate
    #    "weight_decay": 1e-5,  # Add regularization
    #}
)

INFO     Training for 100 epochs.                                                                                  


/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scvi/train/_trainrunner.py:98: UserWarning: `accelerator` has been set to `mps`. Please note that not all PyTorch/Jax operations are supported with this backend. as a result, some models might be slower and less accurate than usual. Please verify your analysis!Refer to https://github.com/pytorch/pytorch/issues/77764 for more details.
  accelerator, lightning_devices, device = parse_device_args(
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf

Epoch 100/100: 100%|██████████| 100/100 [31:00<00:00, 19.44s/it, v_num=1, train_loss=654]

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 100/100: 100%|██████████| 100/100 [31:00<00:00, 18.60s/it, v_num=1, train_loss=654]


In [30]:
adata.obsm['X_scanvi'] = scanvi_model.get_latent_representation()

In [31]:
scanvi_model.save(str(DATA_OUT / 'model_scanvi_obj_a_scanvi'), overwrite=True)
adata.write_h5ad(DATA_OUT / 'integrated_obj_a_scanvi.h5ad', compression='gzip')

In [32]:
sc.pp.neighbors(adata, use_rep='X_scanvi', n_neighbors=50, metric='minkowski')
sc.tl.umap(adata, random_state=0, min_dist=0.2, spread=1.0)

/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scanpy/neighbors/__init__.py:430: FutureWarning: The method obsm_keys is deprecated and will be removed in the future. Use obsm instead of obsm_keys. (e.g. `k in adata.obsm` or `adata.obsm.keys() | {'u'}`)
  if "X_diffmap" in adata.obsm_keys():


In [33]:
COLOR_KEYS = ['Study_name', 'cell_states', 'age_group', 'lgr5_status', 'gut_region']
QC_KEYS    = ['total_counts', 'n_genes_by_counts']

In [36]:
sc.set_figure_params(dpi=300, figsize=(6, 6))
sc.pl.umap(adata, color=[c for c in COLOR_KEYS if c in adata.obs.columns],
           cmap='magma_r', ncols=3, size=2, frameon=False, show=False)
plt.savefig(FIG_DIR / 'umap_metadata.png', bbox_inches='tight'); plt.close()

sc.pl.umap(adata, color=QC_KEYS, cmap='magma_r', ncols=2, size=2,
           frameon=False, show=False)
plt.savefig(FIG_DIR / 'umap_qc.png', bbox_inches='tight'); plt.close()
print('UMAPs saved \u2192', FIG_DIR)


UMAPs saved → /Users/am336941/Library/CloudStorage/OneDrive-GSK/Desktop/Fetal_stem_cells_analysis/analysis_enhanced/3_integration/3a_human_only/figures/obj_a_scanvi


In [37]:
try:
    from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection
    bio = BioConservation(isolated_labels=True, nmi_ari_cluster_labels_kmeans=True,
                          silhouette_label=True, clisi_knn=True)
    batch = BatchCorrection(graph_connectivity=True, ilisi_knn=True,
                            kbet_per_label=False, pcr_comparison=True, bras=True)
    a_lab = adata[adata.obs['cell_states'].astype(str) != 'unknown'].copy()
    bm = Benchmarker(a_lab, batch_key='Study_name', label_key='cell_states',
                     embedding_obsm_keys=['X_scanvi','X_scvi', 'X_pca'],
                     bio_conservation_metrics=bio, batch_correction_metrics=batch, n_jobs=2)
    bm.benchmark()
    res = bm.get_results(min_max_scale=False, clean_names=True)
    print(res)
    res.to_csv(FIG_DIR / 'scib_metrics.csv')
    plt.figure(figsize=(10, 6))
    bm.plot_results_table(min_max_scale=False, show=False)
    plt.savefig(FIG_DIR / 'scib_benchmark.png', dpi=300, bbox_inches='tight')
    plt.close()
except Exception as e:
    print('scIB skipped:', e)

/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(adata, mask_var, use_highly_variable)
/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scanpy/preprocessing/_pca/__init__.py:243: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  Version(ad.__version__) < Version("0.9")
Embeddings:   0%|          | 0/3 [00:00<?, ?it/s]/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scib_metrics/metrics/_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = p

              Isolated labels        KMeans NMI        KMeans ARI  \
Embedding                                                           
X_scanvi             0.610487          0.601036           0.45773   
X_scvi               0.563557          0.264208          0.157273   
X_pca                0.587344          0.038401          0.015442   
Metric Type  Bio conservation  Bio conservation  Bio conservation   

             Silhouette label             cLISI              BRAS  \
Embedding                                                           
X_scanvi              0.55078          0.998967          0.880146   
X_scvi                0.50486          0.911424          0.927911   
X_pca                 0.41283          0.923039          0.641541   
Metric Type  Bio conservation  Bio conservation  Batch correction   

                        iLISI Graph connectivity    PCR comparison  \
Embedding                                                            
X_scanvi             0.077831 

<Figure size 3000x1800 with 0 Axes>